**setup** (drive, kaggle, wandb)

In [1]:
import os
from google.colab import userdata, drive

# Google Drive
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/fer_challenge/'
os.makedirs(SAVE_DIR, exist_ok=True)

# Kaggle
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_KEY')
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')

# WANDB
!pip install wandb -q

import wandb
wandb.login(key=userdata.get('WANDB_API_KEY'))

print("Setup completed successfully.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: akeke23 (akeke23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Setup completed successfully.


**data download**

In [2]:
!pip install -q --upgrade kaggle

!kaggle competitions download -c challenges-in-representation-learning-facial-expression-recognition-challenge
!unzip -q -o challenges-in-representation-learning-facial-expression-recognition-challenge.zip
!ls -la

print("Data ready!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.0/231.0 kB 16.4 MB/s eta 0:00:00
100% 285M/285M [00:01<00:00, 158MB/s]

total 974072
drwxr-xr-x 1 root root      4096 Jun 16 09:31 .
drwxr-xr-x 1 root root      4096 Jun 16 09:08 ..
-rw-r--r-- 1 root root 299063632 Dec 11  2019 challenges-in-representation-learning-facial-expression-recognition-challenge.zip
drwxr-xr-x 4 root root      4096 Jun  4 13:39 .config
drwx------ 5 root root      4096 Jun 16 09:15 drive
-rw-r--r-- 1 root root      7178 Dec 11  2019 example_submission.csv
-rw-r--r-- 1 root root  96433867 Dec 11  2019 fer2013.tar.gz
-rw-r--r-- 1 root root 301072768 Dec 11  2019 icml_face_data.csv
drwxr-xr-x 1 root root      4096 Jun  4 13:39 sample_data
-rw-r--r-- 1 root root  60125203 Dec 11  2019 test.csv
-rw-r--r-- 1 root root 240699943 Dec 11  2019 train.csv
Data ready!


**gpu**

In [3]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Active Device:", device)

Active Device: cuda


**data preprocessing**

In [4]:
EMOTIONS = ['Angry', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']

def parse_pixels(pixel_series):
    raw_arrays = np.array([np.array(p.split(), dtype=np.uint8) for p in pixel_series])
    return raw_arrays.reshape(-1, 48, 48)

# read data and strip spaces
df = pd.read_csv('/content/icml_face_data.csv')
df.columns = df.columns.str.strip()
df['Usage'] = df['Usage'].str.strip()

# create splits
train_mask = df['Usage'] == 'Training'
val_mask   = df['Usage'] == 'PublicTest'
test_mask  = df['Usage'] == 'PrivateTest'

X_train, y_train = parse_pixels(df.loc[train_mask, 'pixels']), df.loc[train_mask, 'emotion'].values
X_val,   y_val   = parse_pixels(df.loc[val_mask, 'pixels']),   df.loc[val_mask, 'emotion'].values
X_test,  y_test  = parse_pixels(df.loc[test_mask, 'pixels']),  df.loc[test_mask, 'emotion'].values

print(f"Data Shapes - Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")

Data Shapes - Train: (28709, 48, 48) | Val: (3589, 48, 48) | Test: (3589, 48, 48)


In [5]:
class EmotionDataset(Dataset):
    def __init__(self, imgs, lbls, transform=None):
        self.imgs = imgs
        self.lbls = lbls
        self.transform = transform

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        # normalization: [0, 255] -> [0, 1]
        tensor_img = torch.from_numpy(self.imgs[idx]).float() / 255.0

        # add channel dimension: (48, 48) -> (1, 48, 48)
        tensor_img = tensor_img.unsqueeze(0)

        if self.transform:
            tensor_img = self.transform(tensor_img)
        return tensor_img, int(self.lbls[idx])

train_data = EmotionDataset(X_train, y_train)
val_data  = EmotionDataset(X_val, y_val)
test_data  = EmotionDataset(X_test, y_test)

# data streaming pipelines
train_loader = DataLoader(train_data, batch_size=64, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_data,   batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_data,  batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

print(f"Loaders ready! Batches per epoch - Train: {len(train_loader)} | Val: {len(val_loader)}")

Loaders ready! Batches per epoch - Train: 449 | Val: 57


**deep cnn**

In [6]:
class DeepCNN(nn.Module):
    def __init__(self, num_classes=7, dropout_rate=0.0, use_batchnorm=False):
        super().__init__()
        self.use_batchnorm = use_batchnorm

        # block 1: 1->32 channels | 48x48 -> 24x24 after pool
        self.conv1 = nn.Conv2d(1,  32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 32, kernel_size=3, padding=1)
        self.bn1   = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(2, 2)

        # block 2: 32->64 channels | 24x24 -> 12x12 after pool
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.bn2   = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(2, 2)

        # block 3: 64->128 channels | 12x12 -> 6x6 after pool
        self.conv5 = nn.Conv2d(64,  128, kernel_size=3, padding=1)
        self.conv6 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.bn3   = nn.BatchNorm2d(128)
        self.pool3 = nn.MaxPool2d(2, 2)

        # block 4: 128->256 channels | 6x6 -> 3x3 after pool
        self.conv7 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.conv8 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.bn4   = nn.BatchNorm2d(256)
        self.pool4 = nn.MaxPool2d(2, 2)

        # fully connected: 256*3*3 -> 512 -> 256 -> 7
        self.fc1    = nn.Linear(256 * 3 * 3, 512)
        self.bn_fc1 = nn.BatchNorm1d(512)
        self.fc2    = nn.Linear(512, 256)
        self.bn_fc2 = nn.BatchNorm1d(256)
        self.out    = nn.Linear(256, num_classes)

        self.drop = nn.Dropout(p=dropout_rate)

    def _conv_block(self, x, conv_a, conv_b, bn, pool):
        # two convolutions with ReLU, optional BN, then pooling
        x = F.relu(conv_a(x))
        x = F.relu(conv_b(x))
        if self.use_batchnorm:
            x = bn(x)
        return pool(x)

    def forward(self, x):
        x = self._conv_block(x, self.conv1, self.conv2, self.bn1, self.pool1)
        x = self._conv_block(x, self.conv3, self.conv4, self.bn2, self.pool2)
        x = self._conv_block(x, self.conv5, self.conv6, self.bn3, self.pool3)
        x = self._conv_block(x, self.conv7, self.conv8, self.bn4, self.pool4)

        x = x.flatten(1)  # (B, 256*3*3)

        x = F.relu(self.fc1(x))
        if self.use_batchnorm:
            x = self.bn_fc1(x)
        x = self.drop(x)

        x = F.relu(self.fc2(x))
        if self.use_batchnorm:
            x = self.bn_fc2(x)
        x = self.drop(x)

        return self.out(x)

# count trainable parameters
m = DeepCNN(use_batchnorm=True, dropout_rate=0.4)
total = sum(p.numel() for p in m.parameters() if p.requires_grad)
print(f'DeepCNN total params: {total:,}')

DeepCNN total params: 2,487,463


In [7]:
import math

test_model = DeepCNN(dropout_rate=0.0, use_batchnorm=False).to(device)
criterion  = nn.CrossEntropyLoss()

images, labels = next(iter(train_loader))
images, labels = images.to(device), labels.to(device)

print('FORWARD CHECK')
with torch.no_grad():
    out       = test_model(images)
    init_loss = criterion(out, labels)
print(f'Initial loss:   {init_loss.item():.4f}')
print(f'Expected (ln7): {math.log(7):.4f}')
print(f'Difference:     {abs(init_loss.item() - math.log(7)):.4f}  (< 0.1 = OK)')

print('\nBACKWARD CHECK')
opt = torch.optim.Adam(test_model.parameters(), lr=1e-3)
for step in range(300):
    opt.zero_grad()
    out  = test_model(images)
    loss = criterion(out, labels)
    loss.backward()
    opt.step()
    if step % 50 == 0 or step == 299:
        acc        = (out.argmax(1) == labels).float().mean().item()
        grad_norm  = sum(p.grad.data.norm(2).item() ** 2
                         for p in test_model.parameters() if p.grad is not None) ** 0.5
        print(f'Step {step:3d} | Loss: {loss.item():.4f} | Acc: {acc:.3f} | GradNorm: {grad_norm:.3f}')

FORWARD CHECK
Initial loss:   1.9396
Expected (ln7): 1.9459
Difference:     0.0063  (< 0.1 = OK)

BACKWARD CHECK
Step   0 | Loss: 1.9396 | Acc: 0.078 | GradNorm: 0.249
Step  50 | Loss: 1.5144 | Acc: 0.359 | GradNorm: 2.076
Step 100 | Loss: 0.3773 | Acc: 0.875 | GradNorm: 0.968
Step 150 | Loss: 0.0036 | Acc: 1.000 | GradNorm: 0.466
Step 200 | Loss: 0.0000 | Acc: 1.000 | GradNorm: 0.000
Step 250 | Loss: 0.0000 | Acc: 1.000 | GradNorm: 0.000
Step 299 | Loss: 0.0000 | Acc: 1.000 | GradNorm: 0.000


**helper functions**

In [8]:
class EarlyStopping:
    def __init__(self, patience=7, min_delta=0.001):
        self.patience     = patience
        self.min_delta    = min_delta
        self.best_acc     = 0.0
        self.counter      = 0
        self.best_weights = None
        self.should_stop  = False

    def step(self, val_acc, model):
        if val_acc > self.best_acc + self.min_delta:
            self.best_acc     = val_acc
            self.counter      = 0
            self.best_weights = {k: v.cpu().clone()
                                 for k, v in model.state_dict().items()}
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True

    def restore_best(self, model):
        # roll model back to best checkpoint after early stop
        if self.best_weights:
            model.load_state_dict(self.best_weights)

In [9]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total, total_grad_norm = 0.0, 0, 0, 0.0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()

        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        total_grad_norm += grad_norm.item()

        optimizer.step()
        total_loss += loss.item() * images.size(0)
        correct    += (outputs.argmax(1) == labels).sum().item()
        total      += labels.size(0)

    return total_loss / total, correct / total, total_grad_norm / len(loader)


def evaluate_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    with torch.inference_mode():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss    = criterion(outputs, labels)
            total_loss += loss.item() * images.size(0)
            correct    += (outputs.argmax(1) == labels).sum().item()
            total      += labels.size(0)

    return total_loss / total, correct / total


def run_experiment(model, run_name, config, train_loader, val_loader,
                   device, save_dir=SAVE_DIR, use_early_stopping=False):

    wandb.init(
        project='fer-challenge',
        name=run_name,
        group=config['architecture'],
        config=config,
        reinit=True
    )

    model     = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config['lr'],
        weight_decay=config.get('weight_decay', 0.0)
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=5
    )

    # early stopping - when it is requried
    stopper = EarlyStopping(patience=7, min_delta=0.001) if use_early_stopping else None
    best_val_acc = 0.0

    for epoch in range(1, config['epochs'] + 1):
        train_loss, train_acc, grad_norm = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss,   val_acc              = evaluate_epoch(model, val_loader, criterion, device)

        scheduler.step(val_acc)
        current_lr = optimizer.param_groups[0]['lr']

        gap = train_acc - val_acc

        log_dict = {
            'epoch':      epoch,
            'train_loss': train_loss,
            'val_loss':   val_loss,
            'train_acc':  train_acc,
            'val_acc':    val_acc,
            'acc_gap':    gap,
            'grad_norm':  grad_norm,
            'lr':         current_lr,
        }

        if stopper is not None:
            log_dict['early_stop_counter'] = stopper.counter

        wandb.log(log_dict)

        print(f'Ep {epoch:02d}/{config["epochs"]} | '
              f'Train {train_acc*100:.1f}% ({train_loss:.4f}) | '
              f'Val {val_acc*100:.1f}% ({val_loss:.4f}) | '
              f'Gap {gap*100:.1f}% | GradNorm {grad_norm:.3f}')

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), os.path.join(save_dir, f'{run_name}_best.pt'))

        # early stopping check
        if stopper is not None:
            stopper.step(val_acc, model)
            if stopper.should_stop:
                print(f'Early stopping triggered at epoch {epoch} '
                      f'(best val_acc: {stopper.best_acc*100:.2f}%)')
                stopper.restore_best(model)  # roll back to best weights
                break

    wandb.log({'epochs_trained': epoch, 'best_val_acc': best_val_acc})
    wandb.finish()
    print(f'\n-> Best val acc: {best_val_acc*100:.2f}%')
    return best_val_acc

**training**

In [ ]:
config_baseline = {
    'architecture':   'DeepCNN',
    'variant':        'Baseline',
    'lr':             1e-3,
    'batch_size':     64,
    'optimizer':      'Adam',
    'epochs':         30,
    'dropout_rate':   0.0,
    'use_batchnorm':  False,
    'weight_decay':   0.0,
    'early_stopping': False,
}

model_baseline = DeepCNN(dropout_rate=0.0, use_batchnorm=False)
acc_baseline   = run_experiment(
    model_baseline, '02_DeepCNN_Baseline', config_baseline,
    train_loader, val_loader, device,
    use_early_stopping=False
)

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Ep 01/30 | Train 25.0% (1.8118) | Val 24.9% (1.7990) | Gap 0.0% | GradNorm 0.516
Ep 02/30 | Train 29.0% (1.7263) | Val 38.4% (1.5771) | Gap -9.5% | GradNorm 0.912
Ep 03/30 | Train 40.5% (1.5026) | Val 43.3% (1.4334) | Gap -2.8% | GradNorm 1.583
Ep 04/30 | Train 46.0% (1.3737) | Val 47.5% (1.3473) | Gap -1.5% | GradNorm 1.956
Ep 05/30 | Train 50.6% (1.2734) | Val 50.4% (1.2984) | Gap 0.2% | GradNorm 2.190
Ep 06/30 | Train 54.6% (1.1796) | Val 52.0% (1.2532) | Gap 2.6% | GradNorm 2.458
Ep 07/30 | Train 58.5% (1.0954) | Val 54.3% (1.2215) | Gap 4.2% | GradNorm 2.664
Ep 08/30 | Train 62.1% (1.0037) | Val 53.9% (1.2642) | Gap 8.3% | GradNorm 2.957
Ep 09/30 | Train 66.1% (0.9065) | Val 54.1% (1.2864) | Gap 12.1% | GradNorm 3.268
Ep 10/30 | Train 70.4% (0.7891) | Val 53.8% (1.3428) | Gap 16.6% | GradNorm 3.675
Ep 11/30 | Train 74.9% (0.6751) | Val 54.5% (1.4542) | Gap 20.4% | GradNorm 4.171
Ep 12/30 | Train 79.7% (0.5568) | Val 54.6% (1.5832) | Gap 25.1% | GradNorm 4.532
Ep 13/30 | Train 83.0

acc_gap,▂▁▂▂▂▃▃▃▄▄▅▅▆▆▆▇▇▇▇▇▇█████████
best_val_acc,▁
epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
epochs_trained,▁
grad_norm,▁▂▂▃▃▄▄▄▅▅▆▆▇▇▇██████▆▅▅▅▅▆▆▃▁
lr,████████████████████▃▃▃▃▃▃▃▁▁▁
train_acc,▁▁▂▃▃▄▄▄▅▅▆▆▆▇▇▇▇▇▇▇▇█████████
train_loss,██▇▆▆▆▅▅▄▄▄▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁
val_acc,▁▄▅▆▇▇██████▇█████▇███████████
val_loss,▂▂▁▁▁▁▁▁▁▁▁▂▂▂▃▃▃▃▃▄▄▅▆▇▇▇▇▇██
acc_gap,0.44626



-> Best val acc: 55.17%


In [ ]:
config_do03 = {
    'architecture':   'DeepCNN',
    'variant':        'Dropout_0.3',
    'lr':             1e-3,
    'batch_size':     64,
    'optimizer':      'Adam',
    'epochs':         30,
    'dropout_rate':   0.3,
    'use_batchnorm':  False,
    'weight_decay':   0.0,
    'early_stopping': False,
}

model_do03 = DeepCNN(dropout_rate=0.3, use_batchnorm=False)
acc_do03   = run_experiment(
    model_do03, '03_DeepCNN_Dropout03', config_do03,
    train_loader, val_loader, device,
    use_early_stopping=False
)

Ep 01/30 | Train 24.9% (1.8207) | Val 24.9% (1.8106) | Gap -0.1% | GradNorm 0.554
Ep 02/30 | Train 25.1% (1.8139) | Val 24.9% (1.8144) | Gap 0.2% | GradNorm 0.378
Ep 03/30 | Train 27.2% (1.7737) | Val 32.0% (1.6952) | Gap -4.9% | GradNorm 0.575
Ep 04/30 | Train 37.7% (1.5719) | Val 42.0% (1.4573) | Gap -4.3% | GradNorm 1.492
Ep 05/30 | Train 43.4% (1.4446) | Val 45.6% (1.4000) | Gap -2.2% | GradNorm 2.012
Ep 06/30 | Train 46.9% (1.3632) | Val 48.0% (1.3547) | Gap -1.2% | GradNorm 2.286
Ep 07/30 | Train 49.9% (1.2948) | Val 50.4% (1.2973) | Gap -0.5% | GradNorm 2.511
Ep 08/30 | Train 52.5% (1.2390) | Val 52.5% (1.2496) | Gap 0.0% | GradNorm 2.777
Ep 09/30 | Train 54.8% (1.1844) | Val 52.7% (1.2406) | Gap 2.1% | GradNorm 2.928
Ep 10/30 | Train 57.1% (1.1266) | Val 53.5% (1.2387) | Gap 3.6% | GradNorm 3.209
Ep 11/30 | Train 59.6% (1.0687) | Val 53.5% (1.2267) | Gap 6.2% | GradNorm 3.490
Ep 12/30 | Train 61.6% (1.0100) | Val 54.1% (1.2422) | Gap 7.5% | GradNorm 3.824
Ep 13/30 | Train 63.8%

acc_gap,▂▂▁▁▁▂▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▇▇▇▇▇███
best_val_acc,▁
epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
epochs_trained,▁
grad_norm,▁▁▁▂▃▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇█▇▇▇███▇▆
lr,█████████████████████▃▃▃▃▃▃▁▁▁
train_acc,▁▁▁▂▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▆▆▇▇▇▇█████
train_loss,███▇▆▆▆▆▅▅▅▅▄▄▄▄▄▃▃▃▃▃▂▂▂▁▁▁▁▁
val_acc,▁▁▃▅▆▆▇▇▇▇▇███████████████████
val_loss,▃▃▂▂▁▁▁▁▁▁▁▁▁▁▁▁▂▂▂▂▂▃▄▄▅▅▆▆▇█
acc_gap,0.41063



-> Best val acc: 55.75%


In [ ]:
config_do05 = {
    'architecture':   'DeepCNN',
    'variant':        'Dropout_0.5',
    'lr':             1e-3,
    'batch_size':     64,
    'optimizer':      'Adam',
    'epochs':         30,
    'dropout_rate':   0.5,
    'use_batchnorm':  False,
    'weight_decay':   0.0,
    'early_stopping': False,
}

model_do05 = DeepCNN(dropout_rate=0.5, use_batchnorm=False)
acc_do05   = run_experiment(
    model_do05, '04_DeepCNN_Dropout05', config_do05,
    train_loader, val_loader, device,
    use_early_stopping=False
)

Ep 01/30 | Train 24.9% (1.8220) | Val 24.9% (1.8111) | Gap -0.0% | GradNorm 0.560
Ep 02/30 | Train 25.1% (1.8132) | Val 24.9% (1.8130) | Gap 0.2% | GradNorm 0.376
Ep 03/30 | Train 25.1% (1.8125) | Val 24.9% (1.8119) | Gap 0.2% | GradNorm 0.307
Ep 04/30 | Train 25.1% (1.8119) | Val 24.9% (1.8132) | Gap 0.2% | GradNorm 0.273
Ep 05/30 | Train 25.1% (1.8123) | Val 24.9% (1.8119) | Gap 0.2% | GradNorm 0.250
Ep 06/30 | Train 25.1% (1.8114) | Val 24.9% (1.8115) | Gap 0.2% | GradNorm 0.241
Ep 07/30 | Train 25.1% (1.8119) | Val 24.9% (1.8128) | Gap 0.2% | GradNorm 0.228
Ep 08/30 | Train 25.1% (1.8110) | Val 24.9% (1.8114) | Gap 0.2% | GradNorm 0.219
Ep 09/30 | Train 25.1% (1.8113) | Val 24.9% (1.8109) | Gap 0.2% | GradNorm 0.217
Ep 10/30 | Train 25.1% (1.8104) | Val 24.9% (1.8115) | Gap 0.2% | GradNorm 0.211
Ep 11/30 | Train 25.1% (1.8110) | Val 24.9% (1.8110) | Gap 0.2% | GradNorm 0.207
Ep 12/30 | Train 25.1% (1.8106) | Val 24.9% (1.8116) | Gap 0.2% | GradNorm 0.205
Ep 13/30 | Train 25.1% (1.8

acc_gap,▁█████████████████████████████
best_val_acc,▁
epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
epochs_trained,▁
grad_norm,█▅▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
lr,██████▄▄▄▄▄▄▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
train_acc,▁█████████████████████████████
train_loss,█▃▃▂▂▂▂▂▂▁▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_acc,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,▁▇▄█▄▃▇▂▁▃▁▃▁▁▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂
acc_gap,0.00194



-> Best val acc: 24.94%


In [ ]:
config_bn = {
    'architecture':   'DeepCNN',
    'variant':        'BatchNorm_Dropout04',
    'lr':             1e-3,
    'batch_size':     64,
    'optimizer':      'Adam',
    'epochs':         30,
    'dropout_rate':   0.4,
    'use_batchnorm':  True,
    'weight_decay':   0.0,
    'early_stopping': False,
}

model_bn = DeepCNN(dropout_rate=0.4, use_batchnorm=True)
acc_bn   = run_experiment(
    model_bn, '05_DeepCNN_BatchNorm_Dropout04', config_bn,
    train_loader, val_loader, device,
    use_early_stopping=False
)

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Ep 01/30 | Train 29.3% (1.7759) | Val 39.1% (1.5277) | Gap -9.8% | GradNorm 4.239
Ep 02/30 | Train 45.2% (1.4106) | Val 50.0% (1.3044) | Gap -4.8% | GradNorm 2.405
Ep 03/30 | Train 53.0% (1.2310) | Val 54.7% (1.1932) | Gap -1.7% | GradNorm 2.257
Ep 04/30 | Train 57.3% (1.1299) | Val 55.7% (1.1855) | Gap 1.6% | GradNorm 2.328
Ep 05/30 | Train 60.8% (1.0458) | Val 57.6% (1.1175) | Gap 3.3% | GradNorm 2.277
Ep 06/30 | Train 64.5% (0.9476) | Val 58.0% (1.1457) | Gap 6.5% | GradNorm 2.261
Ep 07/30 | Train 68.7% (0.8437) | Val 59.7% (1.1272) | Gap 8.9% | GradNorm 2.278
Ep 08/30 | Train 73.8% (0.7198) | Val 60.2% (1.1275) | Gap 13.5% | GradNorm 2.501
Ep 09/30 | Train 79.0% (0.5854) | Val 57.8% (1.3281) | Gap 21.1% | GradNorm 2.751
Ep 10/30 | Train 83.9% (0.4537) | Val 60.8% (1.3550) | Gap 23.1% | GradNorm 3.014
Ep 11/30 | Train 87.6% (0.3567) | Val 59.5% (1.5431) | Gap 28.1% | GradNorm 3.016
Ep 12/30 | Train 90.4% (0.2767) | Val 61.6% (1.6209) | Gap 28.8% | GradNorm 3.021
Ep 13/30 | Train 91.

acc_gap,▁▂▂▃▃▃▄▄▅▆▆▇▇▇▇▇▇▇████████████
best_val_acc,▁
epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
epochs_trained,▁
grad_norm,█▅▄▄▄▄▄▅▅▆▆▆▆▆▆▆▆▅▄▃▃▃▃▃▂▁▁▁▁▁
lr,█████████████████▃▃▃▃▃▃▁▁▁▁▁▁▁
train_acc,▁▃▃▄▄▅▅▅▆▆▇▇▇▇▇███████████████
train_loss,█▇▆▅▅▅▄▄▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁
val_acc,▁▄▆▆▆▇▇▇▇▇▇██▇▇▇▇█▇█▇▇▇▇█████▇
val_loss,▂▂▁▁▁▁▁▁▂▂▂▃▃▃▄▄▅▄▅▆▆▇▇█▇▇████
acc_gap,0.38942



-> Best val acc: 62.69%


In [ ]:
config_bn_es = {
    'architecture':   'DeepCNN',
    'variant':        'BN_Dropout_ES',
    'lr':             1e-3,
    'batch_size':     64,
    'optimizer':      'Adam',
    'epochs':         30,
    'dropout_rate':   0.4,
    'use_batchnorm':  True,
    'weight_decay':   0.0,
    'early_stopping': True,
}

model_bn_es = DeepCNN(dropout_rate=0.4, use_batchnorm=True)
acc_bn_es   = run_experiment(
    model_bn_es, '06_DeepCNN_ES', config_bn_es,
    train_loader, val_loader, device,
    use_early_stopping=True
)

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Ep 01/30 | Train 25.3% (1.8451) | Val 34.4% (1.6568) | Gap -9.2% | GradNorm 4.168
Ep 02/30 | Train 41.7% (1.4989) | Val 44.4% (1.4550) | Gap -2.8% | GradNorm 2.765
Ep 03/30 | Train 50.8% (1.2753) | Val 51.5% (1.2731) | Gap -0.7% | GradNorm 2.653
Ep 04/30 | Train 56.6% (1.1500) | Val 55.7% (1.1747) | Gap 0.9% | GradNorm 2.611
Ep 05/30 | Train 59.7% (1.0636) | Val 58.0% (1.1165) | Gap 1.7% | GradNorm 2.489
Ep 06/30 | Train 63.6% (0.9711) | Val 59.3% (1.1151) | Gap 4.3% | GradNorm 2.557
Ep 07/30 | Train 67.3% (0.8768) | Val 59.1% (1.1027) | Gap 8.2% | GradNorm 2.754
Ep 08/30 | Train 71.6% (0.7748) | Val 61.9% (1.1124) | Gap 9.7% | GradNorm 2.851
Ep 09/30 | Train 76.3% (0.6504) | Val 61.7% (1.1546) | Gap 14.6% | GradNorm 3.275
Ep 10/30 | Train 80.7% (0.5357) | Val 62.2% (1.2401) | Gap 18.6% | GradNorm 3.408
Ep 11/30 | Train 85.5% (0.4141) | Val 61.7% (1.3584) | Gap 23.8% | GradNorm 3.595
Ep 12/30 | Train 88.5% (0.3265) | Val 62.0% (1.5124) | Gap 26.5% | GradNorm 3.737
Ep 13/30 | Train 90.7

acc_gap,▁▂▂▃▃▃▄▄▅▅▆▇▇▇▇█▇█████
best_val_acc,▁
early_stop_counter,▁▁▁▁▁▁▁▂▁▂▁▂▃▅▆▁▂▃▅▆▇█
epoch,▁▁▂▂▂▃▃▃▄▄▄▅▅▅▆▆▆▇▇▇██
epochs_trained,▁
grad_norm,█▃▃▃▃▃▃▄▅▆▆▇▆▆▇▆▆▅▅▅▄▁
lr,████████████████████▁▁
train_acc,▁▃▃▄▄▅▅▅▆▆▇▇▇▇████████
train_loss,█▇▆▅▅▅▄▄▃▃▂▂▂▂▂▁▁▁▁▁▁▁
val_acc,▁▄▅▆▇▇▇███████████████
+1,...



-> Best val acc: 62.33%


In [ ]:
config_bn_es_1 = {
    'architecture':   'DeepCNN',
    'variant':        'BN_Dropout_WeightDecay',
    'lr':             1e-3,
    'batch_size':     64,
    'optimizer':      'Adam',
    'epochs':         30,
    'dropout_rate':   0.4,
    'use_batchnorm':  True,
    'weight_decay':   1e-4,
    'early_stopping': True,
}

model_bn_es_1 = DeepCNN(dropout_rate=0.4, use_batchnorm=True)
acc_bn_es_1   = run_experiment(
    model_bn_es_1, '07_DeepCNN_WD', config_bn_es_1,
    train_loader, val_loader, device,
    use_early_stopping=True
)

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Ep 01/30 | Train 30.9% (1.7438) | Val 41.8% (1.5049) | Gap -10.9% | GradNorm 4.253
Ep 02/30 | Train 47.3% (1.3676) | Val 48.3% (1.3329) | Gap -1.0% | GradNorm 2.502
Ep 03/30 | Train 53.6% (1.2145) | Val 51.0% (1.2699) | Gap 2.6% | GradNorm 2.273
Ep 04/30 | Train 57.4% (1.1244) | Val 56.6% (1.1467) | Gap 0.8% | GradNorm 2.210
Ep 05/30 | Train 60.8% (1.0514) | Val 56.6% (1.1391) | Gap 4.2% | GradNorm 2.288
Ep 06/30 | Train 63.1% (0.9855) | Val 58.9% (1.1102) | Gap 4.2% | GradNorm 2.284
Ep 07/30 | Train 66.4% (0.9105) | Val 59.9% (1.0949) | Gap 6.5% | GradNorm 2.458
Ep 08/30 | Train 69.0% (0.8336) | Val 58.3% (1.1410) | Gap 10.7% | GradNorm 2.661
Ep 09/30 | Train 72.4% (0.7582) | Val 60.0% (1.1499) | Gap 12.3% | GradNorm 2.919
Ep 10/30 | Train 75.6% (0.6742) | Val 60.5% (1.1688) | Gap 15.1% | GradNorm 3.187
Ep 11/30 | Train 79.1% (0.5850) | Val 60.1% (1.1990) | Gap 19.1% | GradNorm 3.531
Ep 12/30 | Train 82.1% (0.5050) | Val 60.3% (1.2733) | Gap 21.8% | GradNorm 3.795
Ep 13/30 | Train 84.

acc_gap,▁▂▃▃▃▃▄▄▄▅▅▆▆▇▇▇▇▇▇▇▇▇▇███████
best_val_acc,▁
early_stop_counter,▁▁▁▁▁▂▁▁▂▁▁▂▃▁▂▁▂▁▂▃▅▆▇█▁▂▃▅▆▇
epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
epochs_trained,▁
grad_norm,▇▂▁▁▁▁▂▂▃▄▅▆▆▇█▇███████▅▄▄▅▅▅▆
lr,██████████████████████▃▃▃▃▃▃▃▁
train_acc,▁▃▃▄▄▄▅▅▅▆▆▆▇▇▇▇▇▇▇▇▇▇▇███████
train_loss,█▆▆▅▅▅▅▄▄▄▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁
val_acc,▁▃▄▆▆▇▇▇▇▇▇▇▇▇█▇█▇▇▇█▇▇███████
+1,...



-> Best val acc: 62.58%


In [ ]:
config_bn_es_2 = {
    'architecture':   'DeepCNN',
    'variant':        'BN_Dropout_WeightDecay_50ep',
    'lr':             1e-3,
    'batch_size':     64,
    'optimizer':      'Adam',
    'epochs':         50,
    'dropout_rate':   0.4,
    'use_batchnorm':  True,
    'weight_decay':   1e-4,
    'early_stopping': True,
}

model_bn_es_2 = DeepCNN(dropout_rate=0.4, use_batchnorm=True)
acc_bn_es_2  = run_experiment(
    model_bn_es_2, '08_DeepCNN_WD_50ep', config_bn_es_2,
    train_loader, val_loader, device,
    use_early_stopping=True
)

Ep 01/50 | Train 27.4% (1.8067) | Val 36.1% (1.6291) | Gap -8.8% | GradNorm 3.991
Ep 02/50 | Train 44.9% (1.4221) | Val 47.3% (1.3467) | Gap -2.4% | GradNorm 2.698
Ep 03/50 | Train 52.6% (1.2410) | Val 53.7% (1.2130) | Gap -1.1% | GradNorm 2.291
Ep 04/50 | Train 56.8% (1.1423) | Val 54.9% (1.1603) | Gap 1.9% | GradNorm 2.207
Ep 05/50 | Train 59.6% (1.0728) | Val 57.2% (1.1294) | Gap 2.4% | GradNorm 2.267
Ep 06/50 | Train 62.5% (1.0081) | Val 57.4% (1.1133) | Gap 5.1% | GradNorm 2.238
Ep 07/50 | Train 65.1% (0.9387) | Val 59.7% (1.1023) | Gap 5.4% | GradNorm 2.390
Ep 08/50 | Train 68.0% (0.8700) | Val 58.8% (1.1221) | Gap 9.1% | GradNorm 2.540
Ep 09/50 | Train 71.2% (0.7888) | Val 59.6% (1.1048) | Gap 11.6% | GradNorm 2.750
Ep 10/50 | Train 74.0% (0.7179) | Val 58.7% (1.1588) | Gap 15.3% | GradNorm 2.986
Ep 11/50 | Train 77.3% (0.6314) | Val 60.1% (1.1892) | Gap 17.2% | GradNorm 3.399
Ep 12/50 | Train 80.0% (0.5606) | Val 59.8% (1.2270) | Gap 20.1% | GradNorm 3.650
Ep 13/50 | Train 83.1

acc_gap,▁▂▂▃▃▃▃▄▄▅▅▅▆▆▆▇▆▇▇▇██████████████████
best_val_acc,▁
early_stop_counter,▁▁▁▁▁▁▁▁▂▃▅▁▂▁▂▃▅▆▇█▁▂▃▁▂▃▅▆▇█▁▁▂▃▅▆▇█
epoch,▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇███
epochs_trained,▁
grad_norm,▆▄▃▃▃▃▃▃▄▄▅▆▆▇▇████▆▅▆▆▇▆▇▇▇▇▄▂▂▂▂▂▂▃▁
lr,██████████████████▄▄▄▄▄▄▄▄▄▄▂▂▂▂▂▂▂▂▁▁
train_acc,▁▃▃▄▄▄▅▅▅▆▆▆▆▇▇▇▇▇▇███████████████████
train_loss,█▆▆▅▅▅▅▄▄▄▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_acc,▁▄▆▆▇▇▇▇▇▇▇▇█▇▇▇▇▇▇██▇██▇▇████████████
+1,...



-> Best val acc: 62.02%


In [ ]:
config_mix = {
    'architecture':   'DeepCNN',
    'variant':        'BN_Dropout03_WD1e3_ES',
    'lr':             1e-3,
    'batch_size':     64,
    'optimizer':      'Adam',
    'epochs':         50,
    'dropout_rate':   0.3,
    'use_batchnorm':  True,
    'weight_decay':   1e-3,
    'early_stopping': True,
}

model_mix = DeepCNN(dropout_rate=0.3, use_batchnorm=True)
acc_mix   = run_experiment(
    model_mix, '09_DeepCNN_DO03_WD1e3_ES', config_mix,
    train_loader, val_loader, device,
    use_early_stopping=True
)

Ep 01/50 | Train 30.1% (1.7553) | Val 39.6% (1.5615) | Gap -9.5% | GradNorm 3.720
Ep 02/50 | Train 45.7% (1.4026) | Val 45.5% (1.3964) | Gap 0.1% | GradNorm 2.192
Ep 03/50 | Train 51.7% (1.2651) | Val 50.4% (1.2792) | Gap 1.3% | GradNorm 1.953
Ep 04/50 | Train 55.0% (1.1885) | Val 54.0% (1.2098) | Gap 1.1% | GradNorm 2.056
Ep 05/50 | Train 57.1% (1.1362) | Val 55.8% (1.1654) | Gap 1.4% | GradNorm 2.248
Ep 06/50 | Train 58.8% (1.0973) | Val 57.1% (1.1411) | Gap 1.6% | GradNorm 2.452
Ep 07/50 | Train 60.7% (1.0638) | Val 56.1% (1.1580) | Gap 4.5% | GradNorm 2.639
Ep 08/50 | Train 61.2% (1.0412) | Val 55.1% (1.1686) | Gap 6.1% | GradNorm 2.889
Ep 09/50 | Train 62.5% (1.0133) | Val 58.0% (1.1251) | Gap 4.4% | GradNorm 3.104
Ep 10/50 | Train 63.7% (0.9943) | Val 59.1% (1.0848) | Gap 4.6% | GradNorm 3.368
Ep 11/50 | Train 64.1% (0.9772) | Val 58.7% (1.1251) | Gap 5.5% | GradNorm 3.614
Ep 12/50 | Train 64.7% (0.9584) | Val 57.7% (1.1552) | Gap 7.1% | GradNorm 3.790
Ep 13/50 | Train 65.5% (0.9

acc_gap,▁▄▄▄▄▄▅▅▅▅▅▆▆▅▆▆▆▆▆▆▇▇▇▇█
best_val_acc,▁
early_stop_counter,▁▁▁▁▁▁▁▂▃▁▁▂▃▅▁▂▃▅▁▂▃▅▆▇█
epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
epochs_trained,▁
grad_norm,▄▁▁▁▂▂▂▃▃▃▄▄▅▅▅▅▆▆▆▆▇▇▇▇█
lr,███████████████████████▁▁
train_acc,▁▃▄▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇█
train_loss,█▆▅▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁
val_acc,▁▃▅▆▇▇▇▆▇██▇▇█▇█▇███▇█▇▇█
+1,...



-> Best val acc: 59.91%


**augmentation**

In [ ]:
from torchvision import transforms

train_aug = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
])

aug_train_ds     = EmotionDataset(X_train, y_train, transform=train_aug)
aug_train_loader = DataLoader(aug_train_ds, batch_size=64, shuffle=True,
                              num_workers=2, pin_memory=True)

In [ ]:
config_aug = {
    'architecture':   'DeepCNN',
    'variant':        'Augmentation',
    'lr':             1e-3,
    'batch_size':     64,
    'optimizer':      'Adam',
    'epochs':         50,
    'dropout_rate':   0.4,
    'use_batchnorm':  True,
    'weight_decay':   1e-4,
    'early_stopping': True,
    'augmentation':   True,
}

model_aug = DeepCNN(dropout_rate=0.4, use_batchnorm=True)
acc_aug   = run_experiment(
    model_aug, '10_DeepCNN_Augmentation', config_aug,
    aug_train_loader, val_loader, device,
    use_early_stopping=True
)

Ep 01/50 | Train 26.3% (1.8314) | Val 28.5% (1.8336) | Gap -2.2% | GradNorm 4.489
Ep 02/50 | Train 40.0% (1.5419) | Val 28.5% (1.8320) | Gap 11.5% | GradNorm 2.474
Ep 03/50 | Train 48.0% (1.3594) | Val 49.6% (1.3294) | Gap -1.5% | GradNorm 2.344
Ep 04/50 | Train 51.7% (1.2631) | Val 53.7% (1.2074) | Gap -2.0% | GradNorm 2.162
Ep 05/50 | Train 54.1% (1.2108) | Val 55.8% (1.1534) | Gap -1.7% | GradNorm 2.054
Ep 06/50 | Train 56.0% (1.1623) | Val 57.1% (1.1485) | Gap -1.1% | GradNorm 2.026
Ep 07/50 | Train 57.2% (1.1349) | Val 55.8% (1.1801) | Gap 1.4% | GradNorm 1.984
Ep 08/50 | Train 58.2% (1.1092) | Val 57.3% (1.1596) | Gap 0.9% | GradNorm 1.970
Ep 09/50 | Train 59.1% (1.0879) | Val 59.7% (1.0922) | Gap -0.5% | GradNorm 1.993
Ep 10/50 | Train 60.1% (1.0684) | Val 59.4% (1.0681) | Gap 0.8% | GradNorm 2.034
Ep 11/50 | Train 60.6% (1.0460) | Val 61.0% (1.0414) | Gap -0.4% | GradNorm 1.982
Ep 12/50 | Train 61.2% (1.0368) | Val 60.1% (1.0652) | Gap 1.1% | GradNorm 2.004
Ep 13/50 | Train 61.

acc_gap,▁▂▁▁▂▃▂▃▂▄▃▃▃▄▃▄▄▅▄▄▅▄▄▅▅▅▅▅▅▆▅▆▆▇▇▇▇▇▇█
best_val_acc,▁
early_stop_counter,▁▁▂▁▁▁▂▁▁▂▂▃▁▁▁▃▁▂▃▅▇▁▂▃▅▇▁▂▃▅▇█▁▁▂▇▁▂▃▆
epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇███
epochs_trained,▁
grad_norm,█▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▂▁▂▂▂▂▂▂▂▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄
lr,██████████████████████████████▁▁▁▁▁▁▁▁▁▁
train_acc,▁▃▄▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇██████████
train_loss,█▆▅▄▄▄▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
val_acc,▁▅▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇████▇██████████
+1,...



-> Best val acc: 67.23%


**weight sampling**

In [12]:
counts = pd.Series(y_train).value_counts().sort_index()
print(counts)

0    3995
1     436
2    4097
3    7215
4    4830
5    3171
6    4965
Name: count, dtype: int64


In [13]:
from torch.utils.data import WeightedRandomSampler
from torchvision import transforms

class_counts = torch.tensor([counts[i] for i in range(7)], dtype=torch.float)
class_weights = 1.0 / class_counts
sample_weights = class_weights[torch.tensor(y_train)]

sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

train_aug = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
])

aug_train_ds = EmotionDataset(X_train, y_train, transform=train_aug)
weighted_loader = DataLoader(aug_train_ds, batch_size=64,
                             sampler=sampler,
                             num_workers=2, pin_memory=True)

In [14]:
config_weighted = {
    'architecture':      'DeepCNN',
    'variant':           'BN_Dropout04_WeightedSampling_Aug',
    'lr':                1e-3,
    'batch_size':        64,
    'optimizer':         'Adam',
    'epochs':            50,
    'dropout_rate':      0.4,
    'use_batchnorm':     True,
    'weight_decay':      1e-4,
    'early_stopping':    True,
    'augmentation':      True,
    'weighted_sampling': True,
}

model_weighted = DeepCNN(dropout_rate=0.4, use_batchnorm=True)
acc_weighted   = run_experiment(
    model_weighted, '11_DeepCNN_WeightedSampling', config_weighted,
    weighted_loader, val_loader, device,
    use_early_stopping=True
)

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Ep 01/50 | Train 18.7% (1.9745) | Val 22.5% (1.8833) | Gap -3.8% | GradNorm 4.015
Ep 02/50 | Train 29.8% (1.7662) | Val 38.3% (1.5709) | Gap -8.5% | GradNorm 2.422
Ep 03/50 | Train 43.3% (1.4967) | Val 47.6% (1.4137) | Gap -4.2% | GradNorm 2.500
Ep 04/50 | Train 49.8% (1.3391) | Val 47.6% (1.3686) | Gap 2.2% | GradNorm 2.377
Ep 05/50 | Train 52.6% (1.2548) | Val 50.4% (1.3188) | Gap 2.3% | GradNorm 2.362
Ep 06/50 | Train 55.6% (1.1825) | Val 52.9% (1.2349) | Gap 2.6% | GradNorm 2.437
Ep 07/50 | Train 57.9% (1.1248) | Val 54.1% (1.2275) | Gap 3.8% | GradNorm 2.373
Ep 08/50 | Train 59.6% (1.0829) | Val 56.5% (1.1479) | Gap 3.1% | GradNorm 2.435
Ep 09/50 | Train 60.6% (1.0555) | Val 56.6% (1.1468) | Gap 4.0% | GradNorm 2.372
Ep 10/50 | Train 61.5% (1.0200) | Val 57.6% (1.1199) | Gap 3.9% | GradNorm 2.423
Ep 11/50 | Train 62.8% (0.9962) | Val 58.5% (1.1254) | Gap 4.3% | GradNorm 2.423
Ep 12/50 | Train 63.6% (0.9748) | Val 59.7% (1.1027) | Gap 3.8% | GradNorm 2.413
Ep 13/50 | Train 64.6% (0

acc_gap,▃▁▃▅▅▅▅▅▆▅▆▅▆▆▇▆▆▆▆▆▆▇▇▆▆▇▇▇▇▇▇▇▇▇██████
best_val_acc,▁
early_stop_counter,▁▁▁▁▂▁▁▁▁▁▁▁▁▂▁▂▁▂▁▂▂▃▅▆▇█▁▂▁▂▃▅▁▂▁▂▃▅▆█
epoch,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇██
epochs_trained,▁
grad_norm,█▁▂▁▁▁▁▁▁▁▁▁▁▁▂▂▁▁▂▂▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅
lr,████████████████████████▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▁
train_acc,▁▂▄▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇█████████████
train_loss,█▇▅▅▄▄▄▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
val_acc,▁▄▅▅▅▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇███████████████
+1,...



-> Best val acc: 65.98%
